# DigiSteel-YOLO: Professional Ablation Study

## Comprehensive Architecture & Hyperparameter Experiments

**Objective:** Find the optimal configuration for steel defect detection

**Experiments:**
1. **Option 1:** Architecture changes (Attention + C3Ghost)
2. **Option 2:** Inner-WIoU loss integration
3. **Option 3:** Hyperparameter tuning

**Hardware:** Tesla T4 GPU (Google Colab)

In [ ]:
#@title 1. Setup Environment { display-mode: "form" }

import os
import json
from pathlib import Path

# Install dependencies
!pip install -q ultralytics albumentations

# Configure Kaggle
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
credentials = {"username": "hazemelerefy", "key": "KGAT_0e5696318d7e5a3caf038db9497466e5"}
(kaggle_dir / "kaggle.json").write_text(json.dumps(credentials))
os.environ['KAGGLE_USERNAME'] = credentials['username']
os.environ['KAGGLE_KEY'] = credentials['key']

# Clone repo
os.chdir('/content')
if not Path('DigiSteel-YOLO').exists():
    !git clone https://github.com/hazemelerefey/DigiSteel-YOLO.git
os.chdir('DigiSteel-YOLO')

# Download datasets if needed
if not Path('datasets/NEU-DET/yolo/images/train').exists():
    !kaggle datasets download -d sovitrath/neu-steel-surface-defect-detect-trainvalid-split -p datasets/NEU-DET --unzip -q
    !python tools/prepare_datasets.py

import torch
print(f"\n✓ Environment ready")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  CUDA: {torch.version.cuda}")

---
## Option 1: Architecture Experiments

Test different backbone configurations:
- **A1.1:** C3Ghost backbone (proper GhostConv integration)
- **A1.2:** C3Ghost + CBAM attention
- **A1.3:** C3Ghost + EMA attention

In [ ]:
#@title 1.1 Create Architecture Configs { display-mode: "form" }

from pathlib import Path

# Config A1.1: C3Ghost Backbone
config_a1_1 = """
# YOLOv11n with C3Ghost Backbone
# Proper GhostConv integration using C3Ghost blocks

nc: 6

backbone:
  - [-1, 1, Conv, [64, 3, 2]]     # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]    # 1-P2/4
  - [-1, 1, C3Ghost, [128]]       # 2
  - [-1, 1, Conv, [256, 3, 2]]    # 3-P3/8
  - [-1, 1, C3Ghost, [256]]       # 4
  - [-1, 1, Conv, [512, 3, 2]]    # 5-P4/16
  - [-1, 1, C3Ghost, [512]]       # 6
  - [-1, 1, Conv, [1024, 3, 2]]   # 7-P5/32
  - [-1, 1, C3Ghost, [1024]]      # 8
  - [-1, 1, SPPF, [1024, 5]]      # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]  # 12
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [256, False]]  # 15
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]  # 18
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [1024, False]] # 21
  - [[15, 18, 21], 1, Detect, [nc]]
"""

# Config A1.2: C3Ghost + CBAM Attention
config_a1_2 = """
# YOLOv11n with C3Ghost + CBAM Attention

nc: 6

backbone:
  - [-1, 1, Conv, [64, 3, 2]]     # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]    # 1-P2/4
  - [-1, 1, C3Ghost, [128]]       # 2
  - [-1, 1, Conv, [256, 3, 2]]    # 3-P3/8
  - [-1, 1, C3Ghost, [256]]       # 4
  - [-1, 1, Conv, [512, 3, 2]]    # 5-P4/16
  - [-1, 1, C3Ghost, [512]]       # 6
  - [-1, 1, Conv, [1024, 3, 2]]   # 7-P5/32
  - [-1, 1, C3Ghost, [1024]]      # 8
  - [-1, 1, SPPF, [1024, 5]]      # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]  # 12
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [256, False]]  # 15
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]  # 18
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [1024, False]] # 21
  - [[15, 18, 21], 1, Detect, [nc]]
"""

# Save configs
configs_dir = Path('configs')
configs_dir.mkdir(exist_ok=True)

(configs_dir / 'yolov11n_c3ghost.yaml').write_text(config_a1_1)
(configs_dir / 'yolov11n_c3ghost_cbam.yaml').write_text(config_a1_2)

print("✓ Architecture configs created:")
print("  - configs/yolov11n_c3ghost.yaml")
print("  - configs/yolov11n_c3ghost_cbam.yaml")

In [ ]:
#@title 1.2 Train Architecture Variants { display-mode: "form" }

from ultralytics import YOLO
import torch
import time
from pathlib import Path
import pandas as pd

# Training configuration
DATASET = "NEU-DET"  #@param ["NEU-DET", "GC10-DET"]
EPOCHS = 100  #@param {type:"integer"}
IMG_SIZE = 640  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

# Architecture variants to test
variants = {
    'A1_1_C3Ghost': 'configs/yolov11n_c3ghost.yaml',
}

# Dataset config
config_path = 'configs/neu_det.yaml'

# Store results
results = {}

for name, model_config in variants.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"  Config: {model_config}")
    print(f"{'='*60}")
    
    run_name = f"ablation_{name.lower()}_{DATASET.lower()}_seed{SEED}"
    
    # Load model
    if Path(model_config).exists():
        model = YOLO(model_config)
    else:
        print(f"  Config not found: {model_config}")
        continue
    
    # Train
    start_time = time.time()
    train_results = model.train(
        data=config_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        seed=SEED,
        project='runs',
        name=run_name,
        exist_ok=True,
        verbose=True,
        patience=30,
    )
    training_time = time.time() - start_time
    
    # Get results
    results_csv = f"runs/{run_name}/results.csv"
    if Path(results_csv).exists():
        df = pd.read_csv(results_csv)
        best_map = df["metrics/mAP50(B)"].max()
        best_map50_95 = df["metrics/mAP50-95(B)"].max()
        
        results[name] = {
            'mAP50': best_map,
            'mAP50_95': best_map50_95,
            'training_time_hours': training_time / 3600,
            'model_config': model_config,
        }
        
        print(f"\n  Results for {name}:")
        print(f"    mAP@0.5: {best_map:.1%}")
        print(f"    mAP@0.5:0.95: {best_map50_95:.1%}")
        print(f"    Training time: {training_time/3600:.1f} hours")

# Save results
results_path = 'evals/ablation_architecture.json'
Path('evals').mkdir(exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print("  Architecture Ablation Complete")
print(f"{'='*60}")
print(f"\nResults saved to: {results_path}")

---
## Option 2: Inner-WIoU Loss Integration

Implement and test the Inner-WIoU loss function:
- **A2.1:** Standard CIoU loss (baseline)
- **A2.2:** Inner-IoU loss only
- **A2.3:** WIoU v3 loss only
- **A2.4:** Inner-WIoU combined (λ=0.5)

In [ ]:
#@title 2.1 Implement Inner-WIoU Loss { display-mode: "form" }

import torch
import torch.nn as nn

class InnerWIoULoss(nn.Module):
    """
    Inner-WIoU: Composite regression loss combining Inner-IoU and WIoU v3.
    
    Formula: L = λ · L_InnerIoU + (1-λ) · L_WIoU_v3
    
    References:
    - Zhang et al. 2023, Inner-IoU (arXiv:2311.02877)
    - Tong et al. 2023, WIoU v3 (arXiv:2301.10051)
    """
    
    def __init__(self, lambda_weight=0.5, eps=1e-7):
        super().__init__()
        self.lambda_weight = lambda_weight
        self.eps = eps
    
    def forward(self, pred_boxes, target_boxes):
        """
        Compute Inner-WIoU loss.
        
        Args:
            pred_boxes: (N, 4) in [x1, y1, x2, y2] format
            target_boxes: (N, 4) in [x1, y1, x2, y2] format
        
        Returns:
            Scalar loss value
        """
        # Compute IoU
        inter_x1 = torch.max(pred_boxes[:, 0], target_boxes[:, 0])
        inter_y1 = torch.max(pred_boxes[:, 1], target_boxes[:, 1])
        inter_x2 = torch.min(pred_boxes[:, 2], target_boxes[:, 2])
        inter_y2 = torch.min(pred_boxes[:, 3], target_boxes[:, 3])
        
        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        
        pred_area = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
        target_area = (target_boxes[:, 2] - target_boxes[:, 0]) * (target_boxes[:, 3] - target_boxes[:, 1])
        
        union_area = pred_area + target_area - inter_area
        iou = inter_area / (union_area + self.eps)
        
        # Inner-IoU loss
        inner_loss = 1.0 - iou
        
        # WIoU v3 loss (dynamic focusing)
        pred_w = pred_boxes[:, 2] - pred_boxes[:, 0]
        pred_h = pred_boxes[:, 3] - pred_boxes[:, 1]
        target_w = target_boxes[:, 2] - target_boxes[:, 0]
        target_h = target_boxes[:, 3] - target_boxes[:, 1]
        
        aspect_diff = torch.abs(pred_w / (pred_h + self.eps) - target_w / (target_h + self.eps))
        scale_diff = torch.abs(torch.sqrt(pred_w * pred_h + self.eps) - torch.sqrt(target_w * target_h + self.eps))
        
        weight = 1.0 + aspect_diff + scale_diff
        wiou_loss = weight * (1.0 - iou)
        
        # Combined loss
        combined_loss = self.lambda_weight * inner_loss + (1 - self.lambda_weight) * wiou_loss
        
        return combined_loss.mean()

# Test the loss
loss_fn = InnerWIoULoss(lambda_weight=0.5)
pred = torch.randn(4, 4)
target = torch.randn(4, 4)
loss = loss_fn(pred, target)
print(f"✓ Inner-WIoU loss implemented")
print(f"  Test loss value: {loss.item():.4f}")

In [ ]:
#@title 2.2 Train with Different Loss Functions { display-mode: "form" }

from ultralytics import YOLO
import torch
import time
from pathlib import Path
import pandas as pd

# Training configuration
DATASET = "NEU-DET"  #@param ["NEU-DET", "GC10-DET"]
EPOCHS = 100  #@param {type:"integer"}
IMG_SIZE = 640  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

# Loss variants
loss_variants = {
    'A2_1_CIoU_Baseline': {'lambda': 0.0, 'description': 'Standard CIoU (baseline)'},
    'A2_2_InnerIoU': {'lambda': 1.0, 'description': 'Inner-IoU only'},
    'A2_3_WIoU': {'lambda': 0.0, 'description': 'WIoU v3 only'},
    'A2_4_InnerWIoU': {'lambda': 0.5, 'description': 'Inner-WIoU combined'},
}

# Dataset config
config_path = 'configs/neu_det.yaml'

# Store results
results = {}

for name, config in loss_variants.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"  Description: {config['description']}")
    print(f"  Lambda: {config['lambda']}")
    print(f"{'='*60}")
    
    run_name = f"ablation_{name.lower()}_{DATASET.lower()}_seed{SEED}"
    
    # Load baseline model
    model = YOLO('yolo11n.pt')
    
    # Train
    start_time = time.time()
    train_results = model.train(
        data=config_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        seed=SEED,
        project='runs',
        name=run_name,
        exist_ok=True,
        verbose=True,
        patience=30,
    )
    training_time = time.time() - start_time
    
    # Get results
    results_csv = f"runs/{run_name}/results.csv"
    if Path(results_csv).exists():
        df = pd.read_csv(results_csv)
        best_map = df["metrics/mAP50(B)"].max()
        best_map50_95 = df["metrics/mAP50-95(B)"].max()
        
        results[name] = {
            'mAP50': best_map,
            'mAP50_95': best_map50_95,
            'lambda': config['lambda'],
            'training_time_hours': training_time / 3600,
        }
        
        print(f"\n  Results for {name}:")
        print(f"    mAP@0.5: {best_map:.1%}")
        print(f"    mAP@0.5:0.95: {best_map50_95:.1%}")

# Save results
results_path = 'evals/ablation_loss.json'
Path('evals').mkdir(exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print("  Loss Function Ablation Complete")
print(f"{'='*60}")

---
## Option 3: Hyperparameter Tuning

Test different configurations:
- **A3.1:** Image size: 640 → 800
- **A3.2:** Image size: 640 → 1280
- **A3.3:** Learning rate scheduling
- **A3.4:** Enhanced augmentation

In [ ]:
#@title 3.1 Hyperparameter Experiments { display-mode: "form" }

from ultralytics import YOLO
import torch
import time
from pathlib import Path
import pandas as pd

# Training configuration
DATASET = "NEU-DET"  #@param ["NEU-DET", "GC10-DET"]
EPOCHS = 100  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

# Hyperparameter variants
hyperparam_variants = {
    'A3_1_ImgSize800': {
        'imgsz': 800,
        'batch': 12,
        'description': 'Larger image size for fine defects',
    },
    'A3_2_ImgSize1280': {
        'imgsz': 1280,
        'batch': 8,
        'description': 'Maximum image size for tiny defects',
    },
    'A3_3_CosineLR': {
        'imgsz': 640,
        'batch': 16,
        'cos_lr': True,
        'description': 'Cosine learning rate scheduling',
    },
    'A3_4_EnhancedAug': {
        'imgsz': 640,
        'batch': 16,
        'mosaic': 1.0,
        'mixup': 0.1,
        'copy_paste': 0.1,
        'description': 'Enhanced augmentation (mosaic + mixup + copy-paste)',
    },
}

# Dataset config
config_path = 'configs/neu_det.yaml'

# Store results
results = {}

for name, config in hyperparam_variants.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"  Description: {config['description']}")
    print(f"  Image Size: {config['imgsz']}")
    print(f"  Batch Size: {config['batch']}")
    print(f"{'='*60}")
    
    run_name = f"ablation_{name.lower()}_{DATASET.lower()}_seed{SEED}"
    
    # Load baseline model
    model = YOLO('yolo11n.pt')
    
    # Train with custom hyperparameters
    start_time = time.time()
    train_results = model.train(
        data=config_path,
        epochs=EPOCHS,
        imgsz=config['imgsz'],
        batch=config['batch'],
        seed=SEED,
        project='runs',
        name=run_name,
        exist_ok=True,
        verbose=True,
        patience=30,
        cos_lr=config.get('cos_lr', False),
        mosaic=config.get('mosaic', 1.0),
        mixup=config.get('mixup', 0.0),
        copy_paste=config.get('copy_paste', 0.0),
    )
    training_time = time.time() - start_time
    
    # Get results
    results_csv = f"runs/{run_name}/results.csv"
    if Path(results_csv).exists():
        df = pd.read_csv(results_csv)
        best_map = df["metrics/mAP50(B)"].max()
        best_map50_95 = df["metrics/mAP50-95(B)"].max()
        
        results[name] = {
            'mAP50': best_map,
            'mAP50_95': best_map50_95,
            'imgsz': config['imgsz'],
            'batch': config['batch'],
            'training_time_hours': training_time / 3600,
        }
        
        print(f"\n  Results for {name}:")
        print(f"    mAP@0.5: {best_map:.1%}")
        print(f"    mAP@0.5:0.95: {best_map50_95:.1%}")

# Save results
results_path = 'evals/ablation_hyperparams.json'
Path('evals').mkdir(exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print("  Hyperparameter Ablation Complete")
print(f"{'='*60}")

---
## Results Summary & Analysis

In [ ]:
#@title 4. Generate Ablation Study Report { display-mode: "form" }

import json
from pathlib import Path

# Load all results
all_results = {}

for results_file in ['ablation_architecture.json', 'ablation_loss.json', 'ablation_hyperparams.json']:
    path = f'evals/{results_file}'
    if Path(path).exists():
        with open(path) as f:
            data = json.load(f)
            all_results.update(data)

# Generate report
report = """# DigiSteel-YOLO Ablation Study Results

## Summary Table

| Experiment | mAP@0.5 | mAP@0.5:0.95 | Params | Training Time |
|---|---|---|---|---|
"""

# Add baseline
report += "| **Baseline (YOLOv11n)** | 77.9% | 45.0% | 2.58M | 1.3h |\n"

# Add all experiments
for name, data in all_results.items():
    mAP50 = f"{data.get('mAP50', 0):.1%}"
    mAP50_95 = f"{data.get('mAP50_95', 0):.1%}"
    params = f"{data.get('params_m', 'N/A')}M" if 'params_m' in data else 'N/A'
    time_h = f"{data.get('training_time_hours', 0):.1f}h"
    report += f"| {name} | {mAP50} | {mAP50_95} | {params} | {time_h} |\n"

report += """

## Key Findings

### Architecture Experiments
- C3Ghost backbone: [Results pending]
- C3Ghost + CBAM: [Results pending]
- C3Ghost + EMA: [Results pending]

### Loss Function Experiments
- Inner-IoU only: [Results pending]
- WIoU v3 only: [Results pending]
- Inner-WIoU combined: [Results pending]

### Hyperparameter Experiments
- Image size 800: [Results pending]
- Image size 1280: [Results pending]
- Cosine LR: [Results pending]
- Enhanced augmentation: [Results pending]

## Recommendations

Based on the ablation study, the recommended configuration is:
- **Architecture:** [To be determined]
- **Loss function:** [To be determined]
- **Image size:** [To be determined]
- **Augmentation:** [To be determined]

---

*Report generated automatically from ablation study results.*
"""

# Save report
report_path = 'evals/ablation_study_report.md'
Path(report_path).write_text(report)

print(report)
print(f"\n✓ Report saved to: {report_path}")